# Python for Machine Learning: Hands-On Engineering Lab

This lab provides practical implementations of essential Python programming patterns, vectorized numerical operations with NumPy, tabular transformations with Pandas, visualization workflows with Matplotlib, and defensive coding techniques for machine learning pipelines.

### Table of Contents
1. [Environment & Imports](#1-environment--imports)
2. [Vectorization vs. Native Loops Benchmark](#2-vectorization-vs-native-loops-benchmark)
3. [NumPy Broadcasting & Tensor Slicing](#3-numpy-broadcasting--tensor-slicing)
4. [Pandas Data Manipulation & Memory Optimization](#4-pandas-data-manipulation--memory-optimization)
5. [Custom ML Dataset Abstraction & Mini-Batch Generator](#5-custom-ml-dataset-abstraction--mini-batch-generator)
6. [Defensive Programming & Exception Handling](#6-defensive-programming--exception-handling)
7. [Publication-Grade Model Evaluation Plots](#7-publication-grade-model-evaluation-plots)

## 1. Environment & Imports
Verify standard dependencies and configure plotting defaults.

In [ ]:
import sys
import time
from dataclasses import dataclass, field
from typing import Generator, Any
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

print(f"Python: {sys.version.split()[0]} | NumPy: {np.__version__} | Pandas: {pd.__version__}")

## 2. Vectorization vs. Native Loops Benchmark
Understanding the execution difference between C-contiguous SIMD vectorization in NumPy and interpreted Python loops.

In [ ]:
# Generate a large synthetic array of 1,000,000 floats
data = np.random.randn(1_000_000).astype(np.float64)

# Approach 1: Native Python Loop
t0 = time.perf_counter()
sum_loop = 0.0
for val in data:
    sum_loop += float(val)
time_loop = time.perf_counter() - t0

# Approach 2: Vectorized NumPy sum
t1 = time.perf_counter()
sum_vec = float(np.sum(data))
time_vec = time.perf_counter() - t1

speedup = time_loop / time_vec
print(f"Loop Time:       {time_loop * 1000:.2f} ms")
print(f"Vectorized Time: {time_vec * 1000:.2f} ms")
print(f"Vectorized Speedup Factor: {speedup:.1f}x")
assert np.isclose(sum_loop, sum_vec, rtol=1e-5)

## 3. NumPy Broadcasting & Tensor Slicing
NumPy compares element shapes from right to left (trailing dimensions first). Two dimensions are compatible if:
1. They are equal, or
2. One of them is 1.

In [ ]:
# Batch of 5 samples with 3 features: Shape (5, 3)
features = np.array([
    [10.0, 2.0, 100.0],
    [15.0, 4.0, 200.0],
    [12.0, 3.0, 150.0],
    [18.0, 5.0, 300.0],
    [20.0, 6.0, 250.0]
])

# Vectorized feature-wise standardization
mu = np.mean(features, axis=0, keepdims=True)  # Shape (1, 3)
sigma = np.std(features, axis=0, keepdims=True)  # Shape (1, 3)

# Broadcasting (5, 3) - (1, 3) -> (5, 3)
standardized = (features - mu) / (sigma + 1e-8)

print("Mean across features:", np.round(np.mean(standardized, axis=0), 4))
print("Std across features: ", np.round(np.std(standardized, axis=0), 4))

## 4. Pandas Data Manipulation & Memory Optimization
Downcasting numeric types to save RAM in data pipelines.

In [ ]:
# Create sample DataFrame with high default memory footprint
np.random.seed(42)
n_rows = 100_000
df = pd.DataFrame({
    "user_id": np.random.randint(1, 1000, size=n_rows, dtype=np.int64),
    "score": np.random.uniform(0.0, 100.0, size=n_rows),
    "category": np.random.choice(["tech", "finance", "health", "education"], size=n_rows)
})

mem_before = df.memory_usage(deep=True).sum() / (1024 * 1024)

# Optimize memory by downcasting
df["user_id"] = df["user_id"].astype(np.int16)
df["score"] = df["score"].astype(np.float32)
df["category"] = df["category"].astype("category")

mem_after = df.memory_usage(deep=True).sum() / (1024 * 1024)
print(f"Initial Memory Usage:   {mem_before:.2f} MB")
print(f"Optimized Memory Usage: {mem_after:.2f} MB")
print(f"Memory Reduction:       {((mem_before - mem_after) / mem_before) * 100:.1f}%")

## 5. Custom ML Dataset Abstraction & Mini-Batch Generator
Demonstrating Python's object-oriented protocols (`__len__`, `__getitem__`) and memory-efficient generators for streaming mini-batches.

In [ ]:
@dataclass
class DatasetConfig:
    name: str
    feature_dim: int
    num_samples: int
    normalize: bool = True

class MLDataset:
    def __init__(self, X: np.ndarray, y: np.ndarray, config: DatasetConfig):
        assert len(X) == len(y), "Feature and target counts must match."
        self.config = config
        self.features = X.astype(np.float32)
        self.targets = y.astype(np.float32)
        if config.normalize:
            mu = np.mean(self.features, axis=0, keepdims=True)
            std = np.std(self.features, axis=0, keepdims=True)
            self.features = (self.features - mu) / (std + 1e-8)

    def __len__(self) -> int:
        return len(self.features)

    def __getitem__(self, idx: int | slice | np.ndarray) -> tuple[np.ndarray, np.ndarray]:
        return self.features[idx], self.targets[idx]

def batch_generator(dataset: MLDataset, batch_size: int, shuffle: bool = True) -> Generator:
    n = len(dataset)
    indices = np.random.permutation(n) if shuffle else np.arange(n)
    for start_idx in range(0, n, batch_size):
        end_idx = min(start_idx + batch_size, n)
        yield dataset[indices[start_idx:end_idx]]

# Instantiate and iterate over batches
X_synth = np.random.randn(100, 4)
y_synth = np.random.randint(0, 2, size=100)
cfg = DatasetConfig(name="synthetic_binary", feature_dim=4, num_samples=100)
dataset = MLDataset(X_synth, y_synth, cfg)

print(f"Loaded dataset with {len(dataset)} samples.")
for step, (bx, by) in enumerate(batch_generator(dataset, batch_size=32)):
    print(f"Batch {step}: X shape = {bx.shape}, y shape = {by.shape}")

## 6. Defensive Programming & Exception Handling
Validating tensor shapes, NaNs, and data types before feeding them into computational pipelines.

In [ ]:
def safe_matrix_inverse(A: np.ndarray) -> np.ndarray:
    """
    Computes matrix inverse with explicit defensive checks.
    """
    if not isinstance(A, np.ndarray):
        raise TypeError(f"Expected np.ndarray, received {type(A).__name__}")
    if A.ndim != 2 or A.shape[0] != A.shape[1]:
        raise ValueError(f"Matrix must be square 2D, received shape {A.shape}")
    if np.isnan(A).any() or np.isinf(A).any():
        raise ValueError("Matrix contains NaN or Inf values")
    
    # Check condition number to detect near-singular matrices
    cond = np.linalg.cond(A)
    if cond > 1e12:
        raise np.linalg.LinAlgError(f"Matrix is ill-conditioned (cond={cond:.2e}), inversion is numerically unstable")
    
    return np.linalg.inv(A)

# Sanity test with well-conditioned matrix
A_valid = np.array([[3.0, 1.0], [1.0, 2.0]])
inv_A = safe_matrix_inverse(A_valid)
print("A inverse:")
print(inv_A)
assert np.allclose(A_valid @ inv_A, np.eye(2))

## 7. Publication-Grade Model Evaluation Plots
Demonstrating clean aesthetic visualization of loss curves and confusion matrices.

In [ ]:
# Generate realistic loss decay curves
epochs = np.arange(1, 31)
train_loss = 2.5 * np.exp(-0.12 * epochs) + 0.15 + np.random.normal(0, 0.02, size=len(epochs))
val_loss = 2.5 * np.exp(-0.10 * epochs) + 0.28 + np.random.normal(0, 0.03, size=len(epochs))

sns.set_theme(style="whitegrid")
plt.figure(figsize=(9, 4.5))
plt.plot(epochs, train_loss, label="Training Loss", color="#2b5c8f", linewidth=2)
plt.plot(epochs, val_loss, label="Validation Loss", color="#d95f02", linestyle="--", linewidth=2)
plt.title("Model Optimization Dynamics", fontsize=13, fontweight="bold", pad=12)
plt.xlabel("Epoch", fontsize=11)
plt.ylabel("Cross-Entropy Loss", fontsize=11)
plt.legend(frameon=True)
plt.tight_layout()
plt.show()